In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1/config.json
/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1/vocab.json
/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1/tokenizer_config.json
/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1/source.spm
/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1/model.safetensors
/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1/target.spm
/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1/generation_config.json


In [2]:
# =============================================
# CELL 1: Cài đặt thư viện
# =============================================
!pip install -q gradio transformers sentencepiece

In [3]:
# =============================================
# CELL 2: Giải nén model đã fine-tune
# =============================================
import os

# ✅ Sửa đúng tên dataset bạn đã upload: envi-model-final1
ZIP_PATH   = "/kaggle/input/envi-model-final1/envi-model-finalversion1.zip"
EXTRACT_TO = "/kaggle/working/envi-model"

print("⏳ Đang giải nén model...")
os.makedirs(EXTRACT_TO, exist_ok=True)
os.system(f"unzip -q {ZIP_PATH} -d {EXTRACT_TO}")

# Kiểm tra giải nén thành công
files = os.listdir(EXTRACT_TO)
print(f"✅ Giải nén xong! Các file: {files}")

⏳ Đang giải nén model...
✅ Giải nén xong! Các file: []


unzip:  cannot find or open /kaggle/input/envi-model-final1/envi-model-finalversion1.zip, /kaggle/input/envi-model-final1/envi-model-finalversion1.zip.zip or /kaggle/input/envi-model-final1/envi-model-finalversion1.zip.ZIP.


In [4]:
# Kiểm tra xem Kaggle dataset có gì bên trong
import os

for root, dirs, files in os.walk("/kaggle/input/envi-model-final1"):
    for f in files:
        print(os.path.join(root, f))

In [5]:
import os
model_path = '/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1' 
print(os.listdir(model_path)) 

['config.json', 'vocab.json', 'tokenizer_config.json', 'source.spm', 'model.safetensors', 'target.spm', 'generation_config.json']


In [6]:
# =============================================
# CELL 3: Load model đã fine-tune
# =============================================
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_PATH = '/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1'

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)
model.eval()

print(f"✅ Load model thành công!")
print(f"   Device     : {device.upper()}")
print(f"   Parameters : {model.num_parameters():,}")

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Load model thành công!
   Device     : CUDA
   Parameters : 127,122,944


In [7]:
# =============================================
# CELL 4: Hàm dịch thuật
# =============================================
def translate(text: str, num_beams: int, max_length: int) -> str:
    text = text.strip()
    if not text:
        return "⚠ Vui lòng nhập văn bản tiếng Anh."

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=int(max_length),
            num_beams=int(num_beams),
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test thử nhanh trước khi mở giao diện
print(translate("I love machine learning.", 5, 128))
print(translate("Artificial intelligence is changing the world.", 5, 128)) 

Tôi yêu việc học máy móc.
Trí tuệ nhân tạo đang thay đổi thế giới.


In [8]:
# =============================================
# CELL 5: Giao diện Gradio
# =============================================
import gradio as gr

EXAMPLES = [
    ["I love machine learning.", 5, 128],
    ["Artificial intelligence is changing the world.", 5, 128],
    ["Although it was raining, we decided to go out.", 5, 128],
    ["The results of this experiment are very promising.", 5, 128],
    ["Please make sure to submit your assignment on time.", 5, 128],
    ["We are developing a neural machine translation system.", 5, 128],
]

css = """
.gradio-container { max-width: 860px !important; margin: auto !important; }
footer { display: none !important; }
"""

with gr.Blocks(css=css, title="EN→VI Translator") as demo:

    gr.HTML("""
        <div style='text-align:center; padding:16px 0 8px'>
            <h1 style='font-size:1.8rem; font-weight:700; margin-bottom:4px'>
                🌐 English → Vietnamese Translator
            </h1>
            <p style='color:#64748b; font-size:0.9rem'>
                MarianMT fine-tuned on <b>PhoMT</b> · Helsinki-NLP/opus-mt-en-vi
            </p>
        </div>
    """)

    with gr.Row():
        src = gr.Textbox(
            label="🇬🇧 English",
            placeholder="Enter English text here...",
            lines=6
        )
        tgt = gr.Textbox(
            label="🇻🇳 Tiếng Việt",
            placeholder="Bản dịch sẽ hiện ở đây...",
            lines=6,
            interactive=False
        )

    with gr.Row():
        num_beams  = gr.Slider(1, 10, value=5, step=1,
                               label="Num Beams",
                               info="Càng cao càng chính xác, càng chậm")
        max_length = gr.Slider(32, 256, value=128, step=16,
                               label="Max Length",
                               info="Độ dài tối đa bản dịch")

    with gr.Row():
        gr.ClearButton(components=[src, tgt], value="🗑 Clear")
        btn = gr.Button("🔤 Translate", variant="primary")

    gr.Examples(
        examples=EXAMPLES,
        inputs=[src, num_beams, max_length],
        outputs=tgt,
        fn=translate,
        cache_examples=False,
        label="📌 Click để thử ngay"
    )

    with gr.Accordion("ℹ️ Thông tin Model", open=False):
        gr.Markdown(f"""
        | | Chi tiết |
        |---|---|
        | **Base model** | Helsinki-NLP/opus-mt-en-vi (MarianMT) |
        | **Fine-tuned on** | PhoMT — ura-hcmut/PhoMT |
        | **Train samples** | 133.317 cặp câu |
        | **Epochs** | 5 |
        | **Decoding** | Beam Search (num_beams=5) |
        | **Device** | `{device.upper()}` |
        | **Parameters** | `{model.num_parameters():,}` |
        """)

    btn.click(fn=translate, inputs=[src, num_beams, max_length], outputs=tgt)
    src.submit(fn=translate, inputs=[src, num_beams, max_length], outputs=tgt)

demo.launch(share=True, debug=False, show_error=True) 

/tmp/ipykernel_55/780429522.py:20: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="EN→VI Translator") as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://9d151fe600b887fbe4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
